# S2.4 — Dependency Injection

Verify FastAPI `Depends()` pattern with typed `Annotated[]` aliases.

**Spec**: `specs/spec-S2.4-dependency-injection/spec.md`

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. Module Imports & Exports

In [2]:
from src.dependency import (
    get_settings,
    get_database,
    get_db_session,
    SettingsDep,
    DatabaseDep,
    SessionDep,
)

import src.dependency as dep_module

# Check all exports
expected = {"get_settings", "get_database", "get_db_session", "SettingsDep", "DatabaseDep", "SessionDep"}
assert expected == set(dep_module.__all__), f"Missing exports: {expected - set(dep_module.__all__)}"
print(f"__all__ = {dep_module.__all__}")
print("\n\u2713 All exports present")

__all__ = ['DatabaseDep', 'SessionDep', 'SettingsDep', 'get_database', 'get_db_session', 'get_settings']

✓ All exports present


## 2. Settings Dependency

In [3]:
from src.config import Settings

settings = get_settings()
assert isinstance(settings, Settings)
print(f"App title: {settings.app.title}")
print(f"App version: {settings.app.version}")

# Singleton check
assert get_settings() is get_settings()
print("\n\u2713 Settings dependency works (singleton)")

App title: PaperAlchemy
App version: 0.1.0

✓ Settings dependency works (singleton)


## 3. Annotated Type Aliases

In [4]:
# Verify all Annotated aliases have __metadata__ (Depends info)
for name, alias in [("SettingsDep", SettingsDep), ("DatabaseDep", DatabaseDep), ("SessionDep", SessionDep)]:
    assert hasattr(alias, "__metadata__"), f"{name} missing __metadata__"
    print(f"{name}: {alias}")

print("\n\u2713 All Annotated aliases valid")

SettingsDep: typing.Annotated[src.config.Settings, Depends(dependency=<functools._lru_cache_wrapper object at 0x10a27ecf0>, use_cache=True, scope=None)]
DatabaseDep: typing.Annotated[src.db.database.Database, Depends(dependency=<function get_database at 0x1088347c0>, use_cache=True, scope=None)]
SessionDep: typing.Annotated[sqlalchemy.ext.asyncio.session.AsyncSession, Depends(dependency=<function get_db_session at 0x10a2f6700>, use_cache=True, scope=None)]

✓ All Annotated aliases valid


## 4. Integration: FastAPI Route with DI

In [5]:
import httpx
from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient
from unittest.mock import MagicMock

app = FastAPI()

@app.get("/test-di")
async def test_di_route(settings: SettingsDep):
    return {"title": settings.app.title, "version": settings.app.version}

# Override with mock
mock_settings = MagicMock()
mock_settings.app.title = "NotebookTestApp"
mock_settings.app.version = "0.0.1"
app.dependency_overrides[get_settings] = lambda: mock_settings

async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as ac:
    resp = await ac.get("/test-di")
    assert resp.status_code == 200
    data = resp.json()
    assert data["title"] == "NotebookTestApp"
    assert data["version"] == "0.0.1"
    print(f"Response: {data}")

print("\n\u2713 FastAPI dependency injection + override works")

Response: {'title': 'NotebookTestApp', 'version': '0.0.1'}

✓ FastAPI dependency injection + override works


## 5. Database Dependency (Error Path)

In [6]:
# Without initializing the database, get_database should raise
import src.db as db_module

# Save and clear the singleton
saved = db_module._database
db_module._database = None

try:
    get_database()
    assert False, "Should have raised RuntimeError"
except RuntimeError as e:
    print(f"Expected error: {e}")
finally:
    db_module._database = saved

print("\n\u2713 Database dependency raises RuntimeError when not initialised")

Expected error: Database not initialised. Call init_database() first.

✓ Database dependency raises RuntimeError when not initialised
